In [ ]:
import kagglehub

path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

In [ ]:
import tensorflow as tf
import keras

train_ds = keras.utils.image_dataset_from_directory(
    path+"/Training",
    image_size=(224,224),
    batch_size=32,
    label_mode="int"
) 

test_ds = keras.utils.image_dataset_from_directory(
    path+"/Testing",
    image_size=(224,224),
    batch_size=32,
    label_mode="int")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
X_train, y_train, X_test, y_test = [], [], [], []

for image, label in train_ds:
	X_train.append(image)
	y_train.append(label)

for image, label in test_ds:
	X_test.append(image)
	y_test.append(label)

X_train = tf.concat(X_train,axis=0) / 255.0
y_train = tf.concat(y_train,axis=0) 
X_test = tf.concat(X_test,axis=0) / 255.0 
y_test = tf.concat(y_test,axis=0)

In [ ]:
len(X_train), len(y_train), len(X_test), len(y_test)

In [ ]:
import matplotlib.pyplot as mtp

mtp.imshow(X_train[0]), y_train[0]

In [ ]:
from keras.callbacks import EarlyStopping 
from keras import Sequential, layers
from keras.layers import Dense,Conv2D,MaxPooling2D,Flatten,Dropout

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)

model = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(224, 224, 3), padding="same"),
	MaxPooling2D((2,2)),

	Conv2D(64,(3,3),activation="relu"),
	MaxPooling2D((2,2)),

	Flatten(),

	Dense(64,activation="relu"),
	Dropout(0.5),
	Dense(32,activation="relu"),
	Dropout(0.5),
	Dense(4,activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
model.evaluate(X_test,y_test)